In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))
from paths import RAW, PROCESSED, RESULTS, savefig

Land Eligibility Analysis

In [ ]:
!pip install atlite

In [ ]:
import geopandas as gpd
import numpy as np
import rasterio
from atlite.gis import ExclusionContainer, shape_availability
from rasterio.plot import show

Solar Eligible Areas

In [ ]:
# Regions produced by 01_regions_centroids.ipynb
gadm = gpd.read_file(PROCESSED / "gadm_aut_level1.geojson")

landcover_path = (
    RAW
    / "copernicus-glc"
    / "PROBAV_LC100_global_v3.0.1_2019-nrt_Discrete-Classification-map_EPSG-4326-AT.tif"
)

suitable_land_codes = [20, 30, 40, 60]

wdpa_path = RAW / "wdpa" / "WDPA_Oct2022_Public_shp-AUT.tif"

In [ ]:
solar_excluder = ExclusionContainer(crs=3035, res=100)

# protected areas
solar_excluder.add_raster(
    wdpa_path,
    nodata=255
)

# suitable land cover
solar_excluder.add_raster(
    landcover_path,
    codes=suitable_land_codes,
    invert=True,
    nodata=255
)

# Creating the mask
regions_m = gadm.to_crs("EPSG:3035")

solar_availability, solar_transform = shape_availability(
    regions_m.geometry,
    solar_excluder
)

In [ ]:
# Export the eligibility mask for reuse by downstream notebooks (e.g. capacity factor / model building)
solar_mask_path = PROCESSED / "solar_availability_AUT.tif"
solar_data = solar_availability.astype("uint8")

with rasterio.open(
    solar_mask_path,
    "w",
    driver="GTiff",
    height=solar_data.shape[0],
    width=solar_data.shape[1],
    count=1,
    dtype=solar_data.dtype,
    crs="EPSG:3035",
    transform=solar_transform,
) as dst:
    dst.write(solar_data, 1)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 8))

show(
    solar_availability,
    transform=solar_transform,
    ax=ax
)

regions_m.boundary.plot(
    ax=ax,
    color="black",
    linewidth=0.8
)

ax.set_title("Solar PV Eligible Areas in Austria")

savefig(fig, "land_eligibility", "solar_eligible_areas.png")
plt.show()